# Multilingual SER — Notebook 3: HF Spaces App Generator

### Setup
Add **notebook 2** as an input (same way you added notebook 1 to notebook 2).

### What gets written to `/kaggle/working/hf_spaces/`
```
app.py                  Gradio app
model_utils.py          inference helpers
requirements.txt        dependencies
fusion_ser.pt           your trained model
gemaps_mlp.pt           your trained model
metadata.json
config.json
language_scalers.pkl    normalization scalers from nb1
```
Upload all of those files to your HF Space.

In [1]:
# ── Cell 1: Find notebook 2 export ────────────────────────────────────────────
import shutil, json
from pathlib import Path

print('Contents of /kaggle/input:')
for item in sorted(Path('/kaggle/input').iterdir()):
    print(f'  {item.name}')

# Search for fusion_ser.pt — the key export from notebook 2
matches = list(Path('/kaggle/input').rglob('fusion_ser.pt'))
# Also check same session
local = Path('/kaggle/working/models/export/fusion_ser.pt')
if local.exists():
    matches.append(local)

if not matches:
    raise RuntimeError(
        'Could not find fusion_ser.pt from notebook 2.\n'
        'Add notebook 2 as an input (+ Add Input -> Notebook tab).\n'
        'Make sure notebook 2 was committed with Save & Run All.'
    )

EXPORT_SRC = matches[0].parent
print(f'\nFound export: {EXPORT_SRC}')
for f in sorted(EXPORT_SRC.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')

HF_DIR = Path('/kaggle/working/hf_spaces')
HF_DIR.mkdir(exist_ok=True)

Contents of /kaggle/input:
  notebooks

Found export: /kaggle/input/notebooks/senadhithimanya/02-model-training-1/models/export
  config.json  (0.0 MB)
  fusion_ser.pt  (33.4 MB)
  gemaps_mlp.pt  (0.1 MB)
  language_scalers.pkl  (0.0 MB)
  metadata.json  (0.0 MB)


In [2]:
# ── Cell 2: Read metadata so generated code has correct values ─────────────────
with open(EXPORT_SRC / 'metadata.json') as f:
    meta = json.load(f)

EMOTION_LABELS = meta['emotion_labels']
NUM_EMOTIONS   = meta['num_emotions']
GEMAPS_DIM     = meta['gemaps_dim']
WHISPER_DIM    = meta['whisper_dim']
SAMPLE_RATE    = meta['sample_rate']
MAX_DURATION   = meta['max_duration']

print(f'Emotions ({NUM_EMOTIONS}): {EMOTION_LABELS}')
print(f'eGeMAPS dim: {GEMAPS_DIM}  |  Whisper dim: {WHISPER_DIM}')
print(f'Sample rate: {SAMPLE_RATE}  |  Max duration: {MAX_DURATION}s')

Emotions (5): ['neutral', 'happy', 'sad', 'angry', 'fear']
eGeMAPS dim: 88  |  Whisper dim: 384
Sample rate: 16000  |  Max duration: 6.0s


In [3]:
# ── Cell 3: Write model_utils.py ──────────────────────────────────────────────

model_utils_code = '''\
import json, os
import numpy as np
import torch
import torch.nn as nn
import librosa
import opensmile
import joblib
from transformers import WhisperModel, WhisperFeatureExtractor

# Loaded from metadata.json at startup — do not hardcode here
EMOTION_LABELS = None
NUM_EMOTIONS   = None
GEMAPS_DIM     = None
WHISPER_DIM    = None
SAMPLE_RATE    = None
MAX_DURATION   = None
MAX_SAMPLES    = None

_smile      = None
_whisper_fe = None
_scalers    = None
_fusion     = None
_mlp        = None


def get_smile():
    global _smile
    if _smile is None:
        _smile = opensmile.Smile(
            feature_set=opensmile.FeatureSet.eGeMAPSv02,
            feature_level=opensmile.FeatureLevel.Functionals,
        )
    return _smile


def get_whisper_fe():
    global _whisper_fe
    if _whisper_fe is None:
        _whisper_fe = WhisperFeatureExtractor.from_pretrained(
            "openai/whisper-tiny", sampling_rate=SAMPLE_RATE
        )
    return _whisper_fe


class GeMAPS_MLP(nn.Module):
    def __init__(self, in_dim, hidden=128, num_classes=5, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.BatchNorm1d(hidden), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.BatchNorm1d(hidden // 2), nn.Dropout(dropout),
            nn.Linear(hidden // 2, num_classes)
        )
    def forward(self, x):
        return self.net(x)


class FusionSER(nn.Module):
    def __init__(self, num_classes=5, dropout=0.3, gemaps_proj=64, whisper_proj=256):
        super().__init__()
        self.whisper_enc = WhisperModel.from_pretrained("openai/whisper-tiny").encoder
        self.w_proj = nn.Sequential(
            nn.Linear(WHISPER_DIM, whisper_proj), nn.ReLU(), nn.Dropout(dropout)
        )
        self.g_proj = nn.Sequential(
            nn.Linear(GEMAPS_DIM, gemaps_proj), nn.ReLU(), nn.Dropout(dropout)
        )
        self.classifier = nn.Sequential(
            nn.Linear(whisper_proj + gemaps_proj, 128), nn.ReLU(),
            nn.BatchNorm1d(128), nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, whisper_inp, gemaps):
        w = self.whisper_enc(whisper_inp).last_hidden_state.mean(dim=1)
        w = self.w_proj(w)
        g = self.g_proj(gemaps)
        return self.classifier(torch.cat([w, g], dim=-1))


def load_models(model_dir="."):
    global _fusion, _mlp, _scalers
    global EMOTION_LABELS, NUM_EMOTIONS, GEMAPS_DIM, WHISPER_DIM
    global SAMPLE_RATE, MAX_DURATION, MAX_SAMPLES

    with open(os.path.join(model_dir, "metadata.json")) as f:
        meta = json.load(f)

    EMOTION_LABELS = meta["emotion_labels"]
    NUM_EMOTIONS   = meta["num_emotions"]
    GEMAPS_DIM     = meta["gemaps_dim"]
    WHISPER_DIM    = meta["whisper_dim"]
    SAMPLE_RATE    = meta["sample_rate"]
    MAX_DURATION   = meta["max_duration"]
    MAX_SAMPLES    = int(SAMPLE_RATE * MAX_DURATION)

    _fusion = FusionSER(num_classes=NUM_EMOTIONS)
    _fusion.load_state_dict(
        torch.load(os.path.join(model_dir, "fusion_ser.pt"), map_location="cpu")
    )
    _fusion.eval()

    _mlp = GeMAPS_MLP(in_dim=GEMAPS_DIM, num_classes=NUM_EMOTIONS)
    _mlp.load_state_dict(
        torch.load(os.path.join(model_dir, "gemaps_mlp.pt"), map_location="cpu")
    )
    _mlp.eval()

    _scalers = joblib.load(os.path.join(model_dir, "language_scalers.pkl"))

    # Pre-warm feature extractors
    get_smile()
    get_whisper_fe()
    print("All models loaded.")


def extract_gemaps(audio_path, language):
    try:
        feats = get_smile().process_file(audio_path)
        arr   = feats.values[0].astype(np.float32).reshape(1, -1)
    except Exception:
        arr = np.zeros((1, GEMAPS_DIM), dtype=np.float32)
    # Apply the same per-language scaler fitted in notebook 1
    scaler = _scalers.get(language) or _scalers.get("english")
    arr    = scaler.transform(arr)
    return torch.from_numpy(arr.astype(np.float32))  # (1, 88)


def extract_whisper(audio_path):
    try:
        audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
        audio    = audio[:MAX_SAMPLES]
        inp      = get_whisper_fe()(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt")
        return inp.input_features  # (1, 80, 3000)
    except Exception:
        return torch.zeros(1, 80, 3000)


@torch.no_grad()
def predict(audio_path, language="english", mode="fusion"):
    if _fusion is None:
        raise RuntimeError("Call load_models() first.")

    gemaps  = extract_gemaps(audio_path, language)
    whisper = extract_whisper(audio_path) if mode in ("fusion", "ensemble") else None

    probs_f = probs_m = None

    if mode in ("fusion", "ensemble"):
        probs_f = torch.softmax(_fusion(whisper, gemaps), -1).squeeze(0).numpy()
    if mode in ("gemaps", "ensemble"):
        probs_m = torch.softmax(_mlp(gemaps), -1).squeeze(0).numpy()

    if mode == "fusion":
        probs = probs_f
    elif mode == "gemaps":
        probs = probs_m
    else:
        probs = 0.6 * probs_f + 0.4 * probs_m

    return {label: float(probs[i]) for i, label in enumerate(EMOTION_LABELS)}
'''

with open(HF_DIR / 'model_utils.py', 'w') as f:
    f.write(model_utils_code)
print('model_utils.py written')

model_utils.py written


In [4]:
# ── Cell 4: Write app.py ──────────────────────────────────────────────────────

# Build these from the actual emotion labels in metadata
emoji  = {'neutral':'😐','happy':'😊','sad':'😢','angry':'😠','fear':'😨'}
colors = {'neutral':'#95a5a6','happy':'#2ecc71','sad':'#3498db','angry':'#e74c3c','fear':'#e67e22'}

emoji_str  = repr({e: emoji.get(e,  '🎙️') for e in EMOTION_LABELS})
colors_str = repr({e: colors.get(e, '#bdc3c7') for e in EMOTION_LABELS})
labels_str = repr(EMOTION_LABELS)

app_code = f'''import gradio as gr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from model_utils import load_models, predict

print("Loading models...")
load_models(model_dir=".")
print("Ready.")

EMOTION_LABELS = {labels_str}
EMOJI          = {emoji_str}
COLORS         = {colors_str}


def run_inference(audio_path, language, mode):
    if audio_path is None:
        return "Please upload or record audio first.", None
    try:
        probs = predict(audio_path, language=language, mode=mode)
    except Exception as e:
        return f"Error: {{e}}", None

    sorted_probs  = sorted(probs.items(), key=lambda x: -x[1])
    top, top_conf = sorted_probs[0]

    result_md = (
        f"## {{EMOJI.get(top, \'\')}} {{top.upper()}}\\n\\n"
        f"**Confidence:** {{top_conf:.1%}}\\n\\n"
        f"**Language:** {{language}}  |  **Mode:** {{mode}}"
    )

    fig, ax = plt.subplots(figsize=(6, 3.2))
    emos  = [e for e, _ in sorted_probs]
    vals  = [p for _, p in sorted_probs]
    cols  = [COLORS.get(e, "#bdc3c7") for e in emos]
    bars  = ax.barh(emos, vals, color=cols, height=0.5, edgecolor="none")
    for bar, val in zip(bars, vals):
        ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{{val:.1%}}", va="center", fontsize=9)
    ax.set_xlim(0, 1.05)
    ax.set_xlabel("Probability")
    ax.set_title("Emotion Probabilities", fontweight="bold")
    ax.invert_yaxis()
    ax.spines[["top", "right", "left"]].set_visible(False)
    plt.tight_layout()

    return result_md, fig


with gr.Blocks(title="Multilingual SER", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # Multilingual Speech Emotion Recognition
    Detects emotion in **Sinhala**, **Tamil**, and **English** speech.
    """)

    with gr.Row():
        with gr.Column():
            audio_in = gr.Audio(
                sources=["upload", "microphone"],
                type="filepath",
                label="Audio Input (WAV/MP3, max 6s)"
            )
            language = gr.Radio(
                choices=["english", "tamil", "sinhala"],
                value="english",
                label="Language",
                info="Select the language spoken — affects normalization"
            )
            mode = gr.Radio(
                choices=["fusion", "gemaps", "ensemble"],
                value="ensemble",
                label="Inference Mode",
                info="ensemble is most robust | gemaps is fastest | fusion is highest accuracy on English/Tamil"
            )
            btn = gr.Button("Detect Emotion", variant="primary")

        with gr.Column():
            out_text = gr.Markdown()
            out_plot = gr.Plot(label="Confidence")

    btn.click(run_inference, [audio_in, language, mode], [out_text, out_plot])

    gr.Markdown("""
    ---
    **Emotions:** Neutral · Happy · Sad · Angry · Fear

    **Modes:**
    - `fusion` — Whisper-tiny encoder + eGeMAPS (best on English & Tamil)
    - `gemaps` — 88 acoustic features only, language-agnostic, ~50ms
    - `ensemble` — 60% fusion + 40% gemaps (recommended for Sinhala)

    > Selecting the correct language is important — the model applies
    > language-specific normalization that was learned during training.
    """)


if __name__ == "__main__":
    demo.launch()
'''

with open(HF_DIR / 'app.py', 'w') as f:
    f.write(app_code)
print('app.py written')

app.py written


In [5]:
# ── Cell 5: Write requirements.txt ────────────────────────────────────────────

reqs = """gradio>=4.0.0
torch>=2.0.0
torchaudio>=2.0.0
transformers>=4.36.0
opensmile>=2.4.0
librosa>=0.10.0
soundfile>=0.12.0
numpy>=1.24.0
scikit-learn>=1.3.0
joblib>=1.3.0
matplotlib>=3.7.0
"""

with open(HF_DIR / 'requirements.txt', 'w') as f:
    f.write(reqs)
print('requirements.txt written')

requirements.txt written


In [6]:
# ── Cell 6: Copy model files ───────────────────────────────────────────────────

to_copy = [
    'fusion_ser.pt',
    'gemaps_mlp.pt',
    'metadata.json',
    'config.json',
    'language_scalers.pkl',
]

for fname in to_copy:
    src = EXPORT_SRC / fname
    dst = HF_DIR / fname
    if src.exists():
        shutil.copy(src, dst)
        print(f'  copied  {fname}  ({dst.stat().st_size/1e6:.1f} MB)')
    else:
        print(f'  MISSING {fname} — check notebook 2 ran fully')

print('\n── Bundle contents ───────────────────────────────')
total = 0
for f in sorted(HF_DIR.iterdir()):
    mb = f.stat().st_size / 1e6
    total += mb
    print(f'  {f.name:<35} {mb:.1f} MB')
print(f'  {"TOTAL":<35} {total:.1f} MB')

  copied  fusion_ser.pt  (33.4 MB)
  copied  gemaps_mlp.pt  (0.1 MB)
  copied  metadata.json  (0.0 MB)
  copied  config.json  (0.0 MB)
  copied  language_scalers.pkl  (0.0 MB)

── Bundle contents ───────────────────────────────
  app.py                              0.0 MB
  config.json                         0.0 MB
  fusion_ser.pt                       33.4 MB
  gemaps_mlp.pt                       0.1 MB
  language_scalers.pkl                0.0 MB
  metadata.json                       0.0 MB
  model_utils.py                      0.0 MB
  requirements.txt                    0.0 MB
  TOTAL                               33.6 MB


In [7]:
# ── Cell 7: Smoke test ────────────────────────────────────────────────────────
# Runs inference locally to catch any mismatch before you upload.

!pip install -q opensmile transformers librosa soundfile joblib

import sys, importlib
sys.path.insert(0, str(HF_DIR))

import model_utils
importlib.reload(model_utils)
from model_utils import load_models, predict

load_models(model_dir=str(HF_DIR))

# Silent clip — just tests the pipeline plumbing
import soundfile as sf
import numpy as np
silent = np.zeros(16000 * 2, dtype=np.float32)
test_wav = '/kaggle/working/smoke_test.wav'
sf.write(test_wav, silent, 16000)

print('Running inference on silent clip...')
for lang in ['english', 'tamil', 'sinhala']:
    for mode in ['gemaps', 'fusion', 'ensemble']:
        result = predict(test_wav, language=lang, mode=mode)
        top = max(result, key=result.get)
        print(f'  [{lang:<10} | {mode:<8}]  {top} ({result[top]:.2%})')

print('\nSmoke test passed — safe to upload to HF Spaces')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.8/137.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.0/325.0 kB 14.7 MB/s eta 0:00:00


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

All models loaded.
Running inference on silent clip...
  [english    | gemaps  ]  sad (88.25%)
  [english    | fusion  ]  neutral (92.20%)
  [english    | ensemble]  neutral (59.51%)
  [tamil      | gemaps  ]  sad (99.99%)
  [tamil      | fusion  ]  neutral (91.70%)
  [tamil      | ensemble]  neutral (55.02%)
  [sinhala    | gemaps  ]  sad (100.00%)
  [sinhala    | fusion  ]  neutral (93.86%)
  [sinhala    | ensemble]  neutral (56.32%)

Smoke test passed — safe to upload to HF Spaces


## Deploying to HF Spaces

1. Go to [huggingface.co/new-space](https://huggingface.co/new-space)
2. **SDK**: Gradio | **Hardware**: CPU Basic (free tier)
3. Upload every file from `/kaggle/working/hf_spaces/`
4. HF Spaces builds automatically — first cold start takes ~90s

### Troubleshooting

| Problem | Fix |
|---------|-----|
| `opensmile` install fails | Pin to `opensmile==2.4.2` |
| Space runs out of memory | Change default mode to `gemaps` in `app.py` |
| Inference times out | Change default mode to `gemaps` (~50ms vs ~2s) |
| Weird predictions | Make sure language is set correctly — scaler is language-specific |